# NLP Lab 4 Task

### **Instructions for Students:**
1. All data preparation, vocabularies, and boilerplate loader functions are pre-written.
2. Do **NOT** copy-paste standard models from the lab demo. Read the specific architectural or algorithmic twists for each task carefully.
3. You must replace all `raise NotImplementedError()` blocks with your code implementation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api
import numpy as np
import random
import urllib.request
import re

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load Pre-trained Google Word2Vec (300-dim)
print("Loading Google Word2Vec Model...")
w2v_google = api.load('word2vec-google-news-300')

def build_embedding_matrix(corpus, w2v_model):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    embedding_dim = w2v_model.vector_size
    weights = [np.zeros(embedding_dim), np.random.randn(embedding_dim) * 0.1]
    
    for sentence in corpus:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
                if word in w2v_model:
                    weights.append(w2v_model[word])
                else:
                    weights.append(np.random.randn(embedding_dim) * 0.1)
                    
    return vocab, torch.tensor(np.array(weights), dtype=torch.float32)

# Named Entity Recognition (NER) with Feature Concatenation
### **The Twist / Challenge:**
Standard sequence labelers rely purely on word embeddings. In Named Entity Recognition (NER), sub-word features like capitalization provide crucial clues (e.g., "Apple" vs "apple").

You must build a BiRNN where the input to the RNN at each time step is the concatenation of:

1. Google Word2Vec embedding vector (300-dim).

2. A custom 1-dimensional feature (1.0 if the word starts with a capital letter, 0.0 otherwise).

In [ ]:
# Data Setup (Provided)

ner_dataset = [
    (["Google", "bought", "YouTube", "in", "USA"], ["ORG", "O", "ORG", "O", "LOC"]),
    (["John", "lives", "in", "London"], ["PER", "O", "O", "LOC"])
]

tag2idx = {"<PAD>": 0, "O": 1, "PER": 2, "ORG": 3, "LOC": 4}
idx2tag = {v: k for k, v in tag2idx.items()}

corpus_t2 = [[w.lower() for w in s[0]] for s in ner_dataset]
vocab_t2, embed_matrix_t2 = build_embedding_matrix(corpus_t2, w2v_google)

def extract_casing_feature(word):
    return [1.0] if word[0].isupper() else [0.0]

# Prepare tensors
max_len_t2 = 5
X_word_ids, X_casing_feats, y_ner = [], [], []

for words, tags in ner_dataset:
    w_ids = [vocab_t2.get(w.lower(), vocab_t2["<UNK>"]) for w in words]
    c_feats = [extract_casing_feature(w) for w in words]
    t_ids = [tag2idx[t] for t in tags]
    
    # Pad
    pad_len = max_len_t2 - len(words)
    w_ids += [vocab_t2["<PAD>"]] * pad_len
    c_feats += [[0.0]] * pad_len
    t_ids += [tag2idx["<PAD>"]] * pad_len
    
    X_word_ids.append(w_ids)
    X_casing_feats.append(c_feats)
    y_ner.append(t_ids)

X_word_tensor = torch.tensor(X_word_ids, dtype=torch.long).to(device)
X_feat_tensor = torch.tensor(X_casing_feats, dtype=torch.float32).to(device) # Shape: (batch, seq_len, 1)
y_ner_tensor = torch.tensor(y_ner, dtype=torch.long).to(device)

In [ ]:
class FeatureConcatBiRNN(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, num_tags):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        
        # TODO 1: Calculate the total input dimension (Word2Vec dimension + 1 extra feature)
        # TODO 2: Initialize Bidirectional RNN with batch_first=True
        # TODO 3: Initialize linear projection layer to map bidirectional hidden states to num_tags
        raise NotImplementedError("Student Task: Define feature concatenated network")

    def forward(self, x_words, x_feats):
        """
        x_words: (batch, seq_len)
        x_feats: (batch, seq_len, 1)
        """
        # TODO 1: Retrieve word embeddings: (batch, seq_len, 300)
        # TODO 2: Concatenate embeddings and x_feats along dimension 2 -> (batch, seq_len, 301)
        # TODO 3: Pass concatenated representation through BiRNN
        # TODO 4: Return logits per token: (batch, seq_len, num_tags)
        raise NotImplementedError("Student Task: Implement forward pass with feature concatenation")

model_t2 = FeatureConcatBiRNN(embed_matrix_t2, hidden_dim=16, num_tags=len(tag2idx)).to(device)
out_t2 = model_t2(X_word_tensor, X_feat_tensor)
print(f"Task 2 Output Shape (Expected: [2, 5, 5]): {out_t2.shape}")